# AI/ML Internship Finder — Build It From Scratch

**Goal:** Build a system that finds Machine Learning Engineer internships and figures out which ones actually fit a student.

By the end you will have a small job-search engine: Python, web data, structured extraction, embeddings, and an LLM.

We do **not** start with an LLM. First we build a basic system that works. Then we add intelligence.

```
                    AI/ML Internship Finder

 Student profile
       │
       ▼
 ┌──────────────┐
 │ Search/Web   │
 │   Sources    │
 └──────┬───────┘
        ▼
 ┌──────────────┐
 │ Collect job  │
 │   postings   │
 └──────┬───────┘
        ▼
 ┌──────────────┐
 │ Extract      │
 │ structured   │
 │ information  │
 └──────┬───────┘
        ▼
 ┌──────────────┐
 │ Match jobs   │◄──── Student profile
 │ to student   │
 └──────┬───────┘
        ▼
 ┌──────────────┐
 │ Rank +       │
 │ explain      │
 └──────┬───────┘
        ▼
   Best internships
```


### You will learn

- Python for data collection
- HTTP and HTML
- CSS selectors
- Web crawling (politely)
- `robots.txt`, pagination, sitemaps, RSS
- When to use an API instead of scraping
- Data cleaning
- Structured data
- LLM extraction
- Embeddings and semantic search
- Ranking
- RAG
- A tiny agent with web tools
- Evaluation

### How to read this notebook

| Marker | Meaning |
|---|---|
| 🟦 **CONCEPT** | Learn the idea |
| 🟩 **BUILD** | Write / run code |
| 🟨 **EXPERIMENT** | Change something and watch what happens |
| 🟧 **AI** | LLM or embeddings |
| 🟥 **CHALLENGE** | You try it |
| 💡 **WHY?** | Why we built it this way |
| 🚨 **IMPORTANT** | Ethics, reliability, rules |

You do **not** need an API key for the first half. If you have a NVIDIA key later, the AI cells turn on. If you don't, those cells fall back to the simpler code you already wrote.


## 1. Setup

🟦 **CONCEPT** — A Python project is just: files on disk, some packages, and a place to put secrets.

Install once in your terminal:

```
pip install -r requirements.txt
```

If you have a NVIDIA key, copy `.env.example` to `.env` and paste it there. If you don't, skip that. The notebook still runs.


🟩 **BUILD** — Import what the basic system needs. No LLM yet.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import time
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

print("imports ok")


What you should see: `imports ok`

The NVIDIA client comes later, only when we need it.


## 2. Define Our Job

🟦 **CONCEPT** — A search system needs a clearly defined retrieval target.

We are not searching for "any internship."
We are searching for Machine Learning Engineer internships.


🟩 **BUILD** — Write the target down as data, not just a sentence in your head.


In [ ]:
TARGET_ROLE = "Machine Learning Engineer Intern"

TARGET_SKILLS = [
    "Python",
    "PyTorch",
    "TensorFlow",
    "Machine Learning",
    "Deep Learning",
]

print("Looking for:", TARGET_ROLE)
print("Skills we care about:", ", ".join(TARGET_SKILLS))


💡 **WHY?** — Later the crawler, the API filter, and the matcher all use this same target. If the target is vague, every later step gets noisier.


## 3. Start With Fake Data

🟦 **CONCEPT** — Before crawling the web, know what a "job record" looks like.

The website is messy. Your database should not be.


🟩 **BUILD** — A tiny dataset with the fields we want by the end.


In [ ]:
jobs = [
    {
        "title": "Machine Learning Intern",
        "company": "Example AI",
        "location": "New York, NY",
        "skills": ["Python", "PyTorch", "Machine Learning"],
        "education": "Bachelor's student",
        "description": "Train models and evaluate them with PyTorch.",
    },
    {
        "title": "Software Engineering Intern",
        "company": "Example Tech",
        "location": "Seattle, WA",
        "skills": ["Java", "React", "AWS"],
        "education": "Bachelor's student",
        "description": "Build web services in Java.",
    },
    {
        "title": "ML Research Intern",
        "company": "Example Research",
        "location": "Boston, MA",
        "skills": ["Python", "TensorFlow", "Machine Learning"],
        "education": "Bachelor's or Master's student",
        "description": "Reproduce papers and run experiments.",
    },
    {
        "title": "Data Analyst Intern",
        "company": "Example Data",
        "location": "Austin, TX",
        "skills": ["SQL", "Excel", "Python"],
        "education": "Bachelor's student",
        "description": "Write SQL and build weekly reports.",
    },
]

df = pd.DataFrame(jobs)
df


What you should see: a 4-row table with title, company, location, skills.

This is the shape the rest of the notebook is trying to produce from real pages.


## 4. HTTP, For Real

🟦 **CONCEPT** — A web page is not magic. It is a URL, a request, a status code, and a pile of text.

```
URL
 ↓
HTTP request
 ↓
status code + HTML
 ↓
BeautifulSoup
 ↓
text / links / tables
```

A URL has pieces:

```
https://campus-jobs.example/jobs/page2.html?page=2
  │         │                  │              │
  scheme    host               path           query
```


🟩 **BUILD** — Split a URL so those pieces are visible.


In [ ]:
sample_url = "https://campus-jobs.example/jobs/page2.html?page=2&q=ml+intern"
parts = urlparse(sample_url)

print("scheme:", parts.scheme)
print("host:  ", parts.netloc)
print("path:  ", parts.path)
print("query: ", parts.query)


💡 **WHY?** — Crawlers spend a lot of time on paths and query strings (`?page=2`). If you cannot read a URL, you will get lost.

🚨 **IMPORTANT** — Web crawling is not "download every website."

Check `robots.txt`, terms of service, rate limits, and whether the site already has an API or a public job feed. Identify yourself. Do not pretend to be a browser to hide.


🟩 **BUILD** — A polite session. The User-Agent says who we are.


In [ ]:
USER_AGENT = "CampusJobFinder/0.1 (student project; classroom use)"

session = requests.Session()
session.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/json;q=0.9,*/*;q=0.8",
})

print(session.headers["User-Agent"])


Status codes you will actually hit:

| Code | Meaning | What the crawler should do |
|---|---|---|
| 200 | OK | Parse the page |
| 301 / 302 | Redirect | Follow it (requests does this by default) |
| 403 | Forbidden | Stop. You are not allowed. |
| 404 | Not found | Skip this URL |
| 429 | Too many requests | Back off. You were too fast. |


🟩 **BUILD** — One real request, just to see a 200. If you are offline, this cell will say so and we keep going.


In [ ]:
try:
    response = session.get("https://example.com", timeout=15)
    print("status:", response.status_code)
    print("content-type:", response.headers.get("Content-Type"))
    print(response.text[:200])
except requests.RequestException as exc:
    print("offline or blocked, skipping live request")
    print(type(exc).__name__)


What you should see: `status: 200` and the start of the example.com HTML.

The rest of class uses a **bundled fake job site** in `data/sample_pages/`. Same skills, no flaky network.


🟩 **BUILD** — Map fake URLs to local files. This is our stand-in for the internet.


In [ ]:
SITE_DIR = Path("data/sample_pages")
FAKE_HOST = "https://campus-jobs.example"


def url_to_path(url):
    path = url.replace(FAKE_HOST, "")
    path = path.split("?")[0].split("#")[0]
    if path in ("", "/"):
        path = "/index.html"
    return SITE_DIR / path.lstrip("/")


def fetch_local(url):
    path = url_to_path(url)
    if not path.exists():
        return {"url": url, "status": 404, "html": "", "from_cache": False}
    html = path.read_text(encoding="utf-8")
    return {"url": url, "status": 200, "html": html, "from_cache": False}


page = fetch_local("https://campus-jobs.example/")
print("status:", page["status"])
print("bytes:", len(page["html"]))
print(page["html"][:250])


What you should see: status 200, and the Campus Jobs homepage HTML.


## 4b. HTML Workshop

🟦 **CONCEPT** — `soup.get_text()` dumps the whole page. That is fine for a first look. It is a bad way to pull a job title.

Real pages use headings, lists, tables, or JSON stuffed in a `<script>` tag. Sometimes they use none of those and you get a pile of `<div>`s.


🟩 **BUILD** — Load the clean posting and pull fields on purpose.


In [ ]:
clean = fetch_local("https://campus-jobs.example/jobs/ml-intern-clean.html")
soup = BeautifulSoup(clean["html"], "html.parser")

print("title:", soup.find("h1").get_text(strip=True))
print()
print("requirements:")
for item in soup.select("ul.req li"):
    print("-", item.get_text(strip=True))


What you should see: `Machine Learning Engineer Intern` and a list that includes Python and PyTorch.


🟨 **EXPERIMENT** — Same job, messy HTML. There is no `<h1>`. The first `<div>` is the site banner.

Run this, then try to find the real title by hand.


In [ ]:
messy = fetch_local("https://campus-jobs.example/jobs/dl-intern-messy.html")
messy_soup = BeautifulSoup(messy["html"], "html.parser")

print("first div:")
print(messy_soup.find("div").get_text(" ", strip=True)[:120])
print()
print("all text, first 400 chars:")
print(messy_soup.get_text(" ", strip=True)[:400])


What you should see: the banner `Harbor AI — Careers — Students...` not the job title. This is why blind `get_text()` lies to you.


🟩 **BUILD** — Skills in a table, and a JobPosting hidden as JSON-LD.


In [ ]:
table_page = fetch_local("https://campus-jobs.example/jobs/ml-research-table.html")
table_soup = BeautifulSoup(table_page["html"], "html.parser")

print("table skills:")
for row in table_soup.select("table tr")[1:]:
    cells_in_row = row.find_all("td")
    if len(cells_in_row) >= 2:
        print("-", cells_in_row[0].get_text(strip=True), "/", cells_in_row[1].get_text(strip=True))

print()

jsonld_page = fetch_local("https://campus-jobs.example/jobs/ml-intern-jsonld.html")
jsonld_soup = BeautifulSoup(jsonld_page["html"], "html.parser")
script = jsonld_soup.find("script", type="application/ld+json")
job_ld = json.loads(script.string)

print("JSON-LD title:", job_ld["title"])
print("JSON-LD company:", job_ld["hiringOrganization"]["name"])
print("JSON-LD skills:", job_ld["skills"])


🟩 **BUILD** — A small cleaner. Drop extra spaces. We will reuse this.


In [ ]:
def clean_text(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def page_text(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    for tag in soup.select("#nav, #footer"):
        tag.decompose()
    return clean_text(soup.get_text(" "))


print(page_text(clean["html"])[:300])


## 4c. Be Allowed To Collect

🚨 **IMPORTANT**

Crawling is not "download the internet."

Before you fetch a URL, ask:

1. Does `robots.txt` allow this path?
2. Do the terms of service allow automated access?
3. Is there an API or a public feed instead?
4. Are you going slowly enough that you are not a problem?
5. Did you already download this page?

If a site needs a login, that is a stop sign for this class — not a puzzle to break.


🟩 **BUILD** — Read the bundled `robots.txt` and ask permission.


In [ ]:
robots = RobotFileParser()
robots.parse((SITE_DIR / "robots.txt").read_text(encoding="utf-8").splitlines())

checks = [
    "https://campus-jobs.example/jobs/page1.html",
    "https://campus-jobs.example/jobs/ml-intern-clean.html",
    "https://campus-jobs.example/private/internal.html",
]

for url in checks:
    allowed = robots.can_fetch(USER_AGENT, url)
    print("allow" if allowed else "block", url)

print("crawl-delay:", robots.crawl_delay(USER_AGENT))


What you should see: `/jobs` allowed, `/private` blocked, crawl-delay `1`.


🟩 **BUILD** — Cache on disk. Fetch once, parse many times.


In [ ]:
CACHE_DIR = Path("data/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def cache_file(url):
    name = hashlib.sha256(url.encode("utf-8")).hexdigest()[:16] + ".html"
    return CACHE_DIR / name


def fetch_page(url, use_cache=True):
    if not robots.can_fetch(USER_AGENT, url):
        return {"url": url, "status": 403, "html": "", "from_cache": False}

    cached = cache_file(url)
    if use_cache and cached.exists():
        return {
            "url": url,
            "status": 200,
            "html": cached.read_text(encoding="utf-8"),
            "from_cache": True,
        }

    page = fetch_local(url)
    if page["status"] == 200:
        cached.write_text(page["html"], encoding="utf-8")
        time.sleep(0.05)  # stand-in for crawl-delay; keep this short in class
    return page


first = fetch_page("https://campus-jobs.example/jobs/ml-intern-clean.html")
second = fetch_page("https://campus-jobs.example/jobs/ml-intern-clean.html")
blocked = fetch_page("https://campus-jobs.example/private/internal.html")

print("first:  status", first["status"], "cache", first["from_cache"])
print("second: status", second["status"], "cache", second["from_cache"])
print("private: status", blocked["status"])


What you should see: first fetch not from cache, second from cache, private `403`.


## 5. Find Internship Pages

🟦 **CONCEPT** — A crawler is a loop.

```
crawler
   ↓
discover links
   ↓
filter relevant links
   ↓
visit pages
   ↓
extract content
```

Most links on a page are junk: nav, email, other sites, `#top`.


🟩 **BUILD** — List every `href` on the homepage. Then we will clean that list.


In [ ]:
home = fetch_page("https://campus-jobs.example/")
home_soup = BeautifulSoup(home["html"], "html.parser")

for link in home_soup.find_all("a"):
    print(repr(link.get("href")), "|", link.get_text(strip=True))


What you should see: useful paths like `/jobs/page1.html` mixed with `mailto:`, `#top`, and `/private`.


🟩 **BUILD** — Turn relative links into full URLs, stay on this host, drop junk.


In [ ]:
def normalize_link(href, base_url):
    if not href:
        return None
    href = href.strip()
    if href.startswith(("mailto:", "javascript:", "#")):
        return None
    full = urljoin(base_url, href)
    full = full.split("#")[0]
    if urlparse(full).netloc != urlparse(FAKE_HOST).netloc:
        return None
    return full


raw_links = [normalize_link(a.get("href"), home["url"]) for a in home_soup.find_all("a")]
links = sorted(set(link for link in raw_links if link))
for link in links:
    print(link)


🟥 **CHALLENGE** — Fill in `is_job_link`.

A job link should look like an internship page, not the homepage, not the blog, not `/private`.

Hint: look at the URL path and the link text.


In [ ]:
def is_job_link(url, link_text=""):
    path = urlparse(url).path.lower()
    text = (link_text or "").lower()
    blob = path + " " + text
    name = path.rsplit("/", 1)[-1]

    if "/private" in path:
        return False
    if path.rstrip("/") in ("", "/index.html"):
        return False
    if name.startswith("page"):
        return False
    if name.endswith((".xml", ".rss", ".json")):
        return False
    # your filter here — try dropping the blog post next
    words = ("intern", "ml-", "swe-")
    return any(word in blob for word in words)


# quick check
print(is_job_link("https://campus-jobs.example/jobs/ml-intern-clean.html", "ML intern"))
print(is_job_link("https://campus-jobs.example/private/internal.html", "Staff only"))
print(is_job_link("https://campus-jobs.example/", "Home"))


## 5b. A Tiny Crawler

🟩 **BUILD** — Frontier in, pages out. Cap it so it cannot run forever.


In [ ]:
def extract_links(html, base_url):
    soup = BeautifulSoup(html, "html.parser")
    found = []
    for tag in soup.find_all("a"):
        url = normalize_link(tag.get("href"), base_url)
        if url:
            found.append((url, tag.get_text(strip=True)))
    return found


def crawl(start_url, max_pages=12):
    frontier = [start_url]
    visited = set()
    pages = []
    log = []

    while frontier and len(visited) < max_pages:
        url = frontier.pop(0)
        if url in visited:
            continue
        path = urlparse(url).path.lower()
        if path.endswith((".xml", ".rss", ".json")):
            visited.add(url)
            continue
        visited.add(url)

        page = fetch_page(url)
        title = ""
        n_links = 0
        if page["status"] == 200:
            soup = BeautifulSoup(page["html"], "html.parser")
            if soup.find("h1"):
                title = soup.find("h1").get_text(strip=True)
            elif soup.find("title"):
                title = soup.find("title").get_text(strip=True)
            links = extract_links(page["html"], url)
            n_links = len(links)
            pages.append({
                "url": url,
                "html": page["html"],
                "text": page_text(page["html"]),
                "title": title,
            })
            for link, text in links:
                if link not in visited and link not in frontier:
                    frontier.append(link)

        log.append({
            "url": url,
            "status": page["status"],
            "title": title,
            "n_links": n_links,
        })

    return pages, pd.DataFrame(log)


crawled_pages, crawl_log = crawl("https://campus-jobs.example/")
print("pages fetched:", len(crawled_pages))
crawl_log


What you should see: a log of URLs, 200s, and at least one 403 for `/private`.

🟥 **CHALLENGE** — Change the loop so it stops once you have 5 pages whose URL looks like a single job posting (`/jobs/` + a file that is not `page1` / `page2`).


## 5c. Pagination

🟦 **CONCEPT** — Job boards almost never put every listing on one HTML file.

Two common patterns:

- a **Next** link
- `?page=2` in the URL


🟩 **BUILD** — Walk page 1 → Next → page 2 and collect listing links.


In [ ]:
def listing_links(list_url):
    page = fetch_page(list_url)
    soup = BeautifulSoup(page["html"], "html.parser")
    jobs_found = []
    next_url = None
    for tag in soup.find_all("a"):
        url = normalize_link(tag.get("href"), list_url)
        if not url:
            continue
        text = tag.get_text(strip=True)
        cls = " ".join(tag.get("class") or [])
        if "next" in cls.lower() or text.lower() == "next":
            next_url = url
        elif is_job_link(url, text):
            jobs_found.append(url)
    return jobs_found, next_url


page_num = 1
list_url = "https://campus-jobs.example/jobs/page1.html"
all_listings = []

while list_url and page_num <= 5:
    found, next_url = listing_links(list_url)
    print("page", page_num, "url", list_url)
    print("  listings:", len(found))
    all_listings.extend(found)
    list_url = next_url
    page_num += 1

all_listings = list(dict.fromkeys(all_listings))
print("unique listings:", len(all_listings))
for url in all_listings:
    print("-", url)


What you should see: page 1 has 3 listings, page 2 has more, including the blog post. Unique count around 6.


## 5d. Sitemaps and RSS

💡 **WHY?** — You do not have to guess links. A lot of sites already publish a map of their pages.


🟩 **BUILD** — Read `sitemap.xml` and `jobs.rss`.


In [ ]:
sitemap_xml = (SITE_DIR / "sitemap.xml").read_text(encoding="utf-8")
sitemap = BeautifulSoup(sitemap_xml, "xml")
sitemap_urls = [loc.get_text(strip=True) for loc in sitemap.find_all("loc")]

print("sitemap")
for url in sitemap_urls:
    print("-", url)

print()

rss_xml = (SITE_DIR / "jobs.rss").read_text(encoding="utf-8")
rss = BeautifulSoup(rss_xml, "xml")

rss_jobs = []
for item in rss.find_all("item"):
    rss_jobs.append({
        "title": item.find("title").get_text(strip=True),
        "url": item.find("link").get_text(strip=True),
        "description": item.find("description").get_text(strip=True),
        "source": "rss",
    })

pd.DataFrame(rss_jobs)


What you should see: sitemap URLs ending in `.html`, and an RSS table with 5 real openings (no blog post).


## 5e. Prefer the API When It Exists

💡 **WHY?** — Structured JSON beats reverse-engineering HTML.

Greenhouse and Lever publish public job boards. No login. No key.

```
GET https://boards-api.greenhouse.io/v1/boards/{token}/jobs?content=true
GET https://api.lever.co/v0/postings/{company}
```

🚨 **IMPORTANT** — These are public feeds, and you should still go slowly. This is not permission to scrape LinkedIn, Indeed, or Handshake.


🟩 **BUILD** — Optional live fetch. Leave `FETCH_LIVE = False` unless you want the network call.


In [ ]:
FETCH_LIVE = False  # set True later if you want real boards


def looks_like_ml_intern(title):
    text = title.lower()
    is_intern = "intern" in text
    tokens = set(re.findall(r"[a-z]+", text))
    is_ml = (
        "machine learning" in text
        or "deep learning" in text
        or bool(tokens & {"ml", "ai"})
    )
    return is_intern and is_ml


def greenhouse_jobs(board_token):
    url = f"https://boards-api.greenhouse.io/v1/boards/{board_token}/jobs"
    r = session.get(url, params={"content": "true"}, timeout=20)
    r.raise_for_status()
    rows = []
    for job in r.json().get("jobs", []):
        title = job.get("title") or ""
        if not looks_like_ml_intern(title):
            continue
        loc = (job.get("location") or {}).get("name", "")
        raw_html = job.get("content") or ""
        rows.append({
            "title": title,
            "company": board_token,
            "location": loc,
            "url": job.get("absolute_url", ""),
            "description": page_text(raw_html) if raw_html else title,
            "source": "greenhouse",
        })
    return rows


def lever_jobs(company):
    url = f"https://api.lever.co/v0/postings/{company}"
    r = session.get(url, timeout=20)
    r.raise_for_status()
    rows = []
    for job in r.json():
        title = job.get("text") or ""
        if not looks_like_ml_intern(title):
            continue
        loc = (job.get("categories") or {}).get("location", "")
        desc = job.get("descriptionPlain") or job.get("description") or ""
        rows.append({
            "title": title,
            "company": company,
            "location": loc,
            "url": job.get("hostedUrl", ""),
            "description": clean_text(desc)[:2000],
            "source": "lever",
        })
    return rows


live_jobs = []
if FETCH_LIVE:
    for token in ["stripe", "databricks"]:
        try:
            found = greenhouse_jobs(token)
            print(token, "greenhouse", len(found))
            live_jobs.extend(found)
        except requests.RequestException as exc:
            print(token, "greenhouse failed:", type(exc).__name__)
        time.sleep(1)
else:
    print("FETCH_LIVE is False — using bundled pages only")

print("live rows:", len(live_jobs))


## 5f. JavaScript-Rendered Pages

🟦 **CONCEPT** — Sometimes "View Source" is empty and the browser is full.

The HTML is a shell:

```html
<div id="app"></div>
```

and the real listings come from `/spa/jobs.json`.

BeautifulSoup cannot run JavaScript. It will tell you the list is empty. That is not a reason to fire up a browser to punch through a login wall. Look for the JSON or the API the page already calls.


🟩 **BUILD** — Prove the HTML is empty, then load the JSON next to it.


In [ ]:
spa = fetch_local("https://campus-jobs.example/spa/index.html")
spa_soup = BeautifulSoup(spa["html"], "html.parser")
print("text inside #app:", repr(spa_soup.select_one("#app").get_text(strip=True)))

spa_json = json.loads((SITE_DIR / "spa" / "jobs.json").read_text(encoding="utf-8"))
pd.DataFrame(spa_json)


What you should see: `#app` is empty, and the JSON table has two internships.

🚨 **IMPORTANT** — We are not using Selenium or Playwright in this class to walk through logins or bot checks. If the data is in a feed, use the feed.


## 5g. One Collector

🟩 **BUILD** — Merge crawl + RSS + optional APIs into one table. This is what the rest of the notebook eats.


In [ ]:
def is_posting_url(url):
    path = urlparse(url).path.lower()
    if "/jobs/" not in path:
        return False
    name = path.rsplit("/", 1)[-1]
    if name.startswith("page"):
        return False
    return name.endswith(".html")


def collect_jobs():
    rows = []

    for page in crawled_pages:
        if not is_posting_url(page["url"]):
            continue
        rows.append({
            "title": page["title"],
            "company": "",
            "location": "",
            "url": page["url"],
            "description": page["text"],
            "html": page["html"],
            "source": "crawl",
        })

    known_urls = {row["url"] for row in rows}
    for item in rss_jobs:
        if item["url"] in known_urls:
            continue
        page = fetch_page(item["url"])
        rows.append({
            "title": item["title"],
            "company": "",
            "location": "",
            "url": item["url"],
            "description": item["description"],
            "html": page.get("html", ""),
            "source": "rss",
        })

    for item in live_jobs:
        rows.append({
            "title": item["title"],
            "company": item.get("company", ""),
            "location": item.get("location", ""),
            "url": item["url"],
            "description": item.get("description", ""),
            "html": "",
            "source": item.get("source", "api"),
        })

    return pd.DataFrame(rows)


collected = collect_jobs()
print(collected["source"].value_counts().to_string())
print("total:", len(collected))
collected[["title", "url", "source"]]


What you should see: several crawl rows (including the blog post — we will deal with that next) and a total count.


## 6. Convert Webpages → Job Records

🟦 **CONCEPT** — A page is a blob of HTML. The database wants this:

```json
{
  "title": "...",
  "company": "...",
  "location": "...",
  "skills": ["..."],
  "education": "...",
  "description": "..."
}
```

First we do this with rules. No LLM.


🟩 **BUILD** — Rule-based extraction. JSON-LD first, then headings, then a table, then a weak text fallback.


In [ ]:
SKILL_BANK = TARGET_SKILLS + [
    "Java", "React", "AWS", "SQL", "Excel", "Python",
]


def extract_jsonld_job(html):
    soup = BeautifulSoup(html, "html.parser")
    tag = soup.find("script", type="application/ld+json")
    if not tag or not tag.string:
        return None
    try:
        data = json.loads(tag.string)
    except json.JSONDecodeError:
        return None
    if data.get("@type") != "JobPosting":
        return None
    loc = data.get("jobLocation", {})
    address = loc.get("address", {}) if isinstance(loc, dict) else {}
    city = address.get("addressLocality", "")
    region = address.get("addressRegion", "")
    location = ", ".join(part for part in [city, region] if part)
    org = data.get("hiringOrganization") or {}
    return {
        "title": data.get("title", ""),
        "company": org.get("name", ""),
        "location": location,
        "skills": list(data.get("skills") or []),
        "education": "",
        "description": clean_text(data.get("description") or ""),
    }


def skills_from_text(text):
    found = []
    lower = text.lower()
    for skill in SKILL_BANK:
        if skill.lower() in lower and skill not in found:
            found.append(skill)
    return found


def extract_job_rules(html, url=""):
    structured = extract_jsonld_job(html)
    if structured:
        structured["url"] = url
        structured["extractor"] = "jsonld"
        return structured

    soup = BeautifulSoup(html, "html.parser")
    title = ""
    if soup.find("h1"):
        title = soup.find("h1").get_text(strip=True)
    elif soup.find("title"):
        title = soup.find("title").get_text(strip=True)

    text = page_text(html)
    company = ""
    location = ""
    education = ""

    company_match = re.search(r"Company:\s*(.+)", text)
    if company_match:
        company = company_match.group(1).split(" Location")[0].strip()

    loc_match = re.search(r"Location:\s*([A-Za-z .]+,\s*[A-Z]{2})", text)
    if loc_match:
        location = loc_match.group(1).strip()

    h1 = soup.find("h1")
    if h1:
        nxt = h1.find_next("p")
        if nxt:
            line = nxt.get_text(" ", strip=True)
            dash = re.match(r"(.+?)\s+[\u2014-]\s+(.+)", line)
            if dash:
                if not company:
                    company = dash.group(1).replace("Company:", "").strip()
                if not location:
                    location = dash.group(2).replace("Location:", "").strip()

    edu_match = re.search(r"Education:\s*(.+?)(?:Requirements|Skills|$)", text)
    if edu_match:
        education = edu_match.group(1).strip()

    skills = []
    for item in soup.select("ul.req li"):
        skills.append(item.get_text(strip=True))
    if not skills:
        for row in soup.select("table tr"):
            tds = row.find_all("td")
            if tds:
                skills.append(tds[0].get_text(strip=True))
    if not skills:
        skills = skills_from_text(text)

    extractor = "rules"
    if not soup.find("h1"):
        extractor = "weak"

    return {
        "title": title,
        "company": company,
        "location": location,
        "skills": skills,
        "education": education,
        "description": text[:1500],
        "url": url,
        "extractor": extractor,
    }


sample_urls = [
    "https://campus-jobs.example/jobs/ml-intern-clean.html",
    "https://campus-jobs.example/jobs/dl-intern-messy.html",
    "https://campus-jobs.example/jobs/ml-research-table.html",
    "https://campus-jobs.example/jobs/ml-intern-jsonld.html",
    "https://campus-jobs.example/jobs/how-i-got-my-internship.html",
]

extracted = []
for url in sample_urls:
    page = fetch_page(url)
    rec = extract_job_rules(page["html"], url)
    extracted.append(rec)
    print(rec["extractor"], "|", rec["title"], "|", rec["company"], "|", rec["skills"])


What you should see:

- clean page → title, company, skills
- JSON-LD page → clean structured fields
- table page → skills from the table
- messy page → weak / incomplete
- blog post → looks like an article, not a job

Rules work sometimes. Webpages are messy. Next we give the hard cases to an LLM.


In [ ]:
job_records = []
for _, row in collected.iterrows():
    html = row.get("html") or ""
    if html:
        rec = extract_job_rules(html, row["url"])
    else:
        rec = {
            "title": row["title"],
            "company": row.get("company", ""),
            "location": row.get("location", ""),
            "skills": skills_from_text(row.get("description", "")),
            "education": "",
            "description": row.get("description", ""),
            "url": row["url"],
            "extractor": "api",
        }
    rec["source"] = row["source"]
    job_records.append(rec)

jobs_df = pd.DataFrame(job_records)
jobs_df[["title", "company", "location", "skills", "extractor", "source"]]


## 7. LLM Structured Extraction

🟧 **AI** — An LLM is not only a chatbot. It can turn a messy page into the same JSON we just defined.

If `NVIDIA_API_KEY` is missing, we keep the rule-based records and move on.


🟩 **BUILD** — Turn the key on if you have one.


In [ ]:
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY", "").strip()
HAS_NVIDIA = bool(NVIDIA_API_KEY)

CHAT_MODEL = "meta/llama-3.1-8b-instruct"
EMBED_MODEL = "nvidia/nv-embedqa-e5-v5"

client = None
if HAS_NVIDIA:
    from openai import OpenAI
    client = OpenAI(
        api_key=NVIDIA_API_KEY,
        base_url="https://integrate.api.nvidia.com/v1",
    )
    print("AI unlocked — NVIDIA client ready")
else:
    print("running without a key — rule-based fallback stays on")


In [ ]:
EXTRACT_PROMPT = '''Read the job posting. Return JSON only. No markdown.
Keys:
  title, company, location, skills, education, experience, is_job_posting
skills is a list of strings.
is_job_posting is true only if this page is an actual opening, not a blog post.

POSTING:
'''


def chat_json(prompt):
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=400,
    )
    text = response.choices[0].message.content.strip()
    text = text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)


def extract_job_llm(html, url=""):
    text = page_text(html)[:3500]
    try:
        data = chat_json(EXTRACT_PROMPT + text)
    except Exception:
        data = extract_job_rules(html, url)
        data["extractor"] = "rules-fallback"
        return data

    data["url"] = url
    data["description"] = text[:1500]
    data["extractor"] = "llm"
    data.setdefault("skills", [])
    data.setdefault("company", "")
    data.setdefault("location", "")
    data.setdefault("education", "")
    return data


def extract_job(html, url=""):
    if HAS_NVIDIA and html:
        return extract_job_llm(html, url)
    return extract_job_rules(html, url)


demo_url = "https://campus-jobs.example/jobs/dl-intern-messy.html"
demo_page = fetch_page(demo_url)
demo_rec = extract_job(demo_page["html"], demo_url)
print(json.dumps(demo_rec, indent=2)[:800])


What you should see: JSON with a title and skills. Without a key, it will look like the weak rule-based result. With a key, the messy Harbor AI page should get a real title.


In [ ]:
# rebuild records with whatever extractor we have
job_records = []
for _, row in collected.iterrows():
    html = row.get("html") or ""
    rec = extract_job(html, row["url"]) if html else {
        "title": row["title"],
        "company": row.get("company", ""),
        "location": row.get("location", ""),
        "skills": skills_from_text(row.get("description", "")),
        "education": "",
        "description": row.get("description", ""),
        "url": row["url"],
        "extractor": "api",
        "is_job_posting": True,
    }
    rec["source"] = row["source"]
    job_records.append(rec)

jobs_df = pd.DataFrame(job_records)

# drop obvious blog posts when the flag exists
if "is_job_posting" in jobs_df.columns:
    mask = jobs_df["is_job_posting"].fillna(True)
    # rule-based path has no flag — keep those rows
    keep = mask | jobs_df["extractor"].isin(["rules", "weak", "jsonld", "api"])
    # still drop the known blog url
    keep = keep & ~jobs_df["url"].str.contains("how-i-got-my-internship")
    jobs_df = jobs_df[keep].reset_index(drop=True)

jobs_df[["title", "company", "location", "skills", "extractor"]]


## 8. Create the Student Profile

🟩 **BUILD** — This is the other side of the match. Edit it so it looks like you.


In [ ]:
student = {
    "name": "Alex",
    "major": "Computer Science",
    "graduation_year": 2027,
    "skills": [
        "Python",
        "Java",
        "Machine Learning",
        "PyTorch",
    ],
    "experience": [
        "ML research project",
        "Python software project",
    ],
    "location_preference": "Boston, MA",
}

student


## 9. A Simple Matching Algorithm

🟦 **CONCEPT** — Do not use AI yet.

If the student has 2 of the 4 skills the job listed, the score is `0.5`.


🟩 **BUILD** — Overlap of skill sets.


In [ ]:
def skill_match(student_skills, job_skills):
    if not job_skills:
        return 0.0
    left = set(s.lower() for s in student_skills)
    right = set(s.lower() for s in job_skills)
    matches = left & right
    return len(matches) / len(right)


for rec in job_records:
    rec["skill_score"] = skill_match(student["skills"], rec.get("skills") or [])

pd.DataFrame(job_records)[["title", "skills", "skill_score"]]


What you should see: the ML internships score higher than the SWE intern. Scores between 0 and 1.


🟨 **EXPERIMENT** — Keyword matching is brittle.

A job can say "neural network frameworks."
The student has "PyTorch."
Those are related. A set overlap will miss it.


In [ ]:
tricky_job_skills = ["neural network frameworks", "Python"]
print("keyword score:", skill_match(student["skills"], tricky_job_skills))
print("student has PyTorch, job never said PyTorch")


💡 **WHY?** — This is the hole embeddings are about to fill.


## 10. Embeddings

🟦 **CONCEPT** — Embeddings turn text into vectors. Nearby vectors mean similar meaning.

```
"PyTorch experience"
        ↕
"experience developing neural networks"
```

Those can be close in vector space even when the words differ.

```
Student profile
      ↓
 embedding
      ↓
vector space
      ↑
 job embedding
      ↑
 internship
```


🟩 **BUILD** — Cosine similarity, plus a bag-of-words fallback so this cell works without a key.


In [ ]:
def cosine(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def tokenize(text):
    return re.findall(r"[a-z0-9]+", (text or "").lower())


def bow_embed(texts):
    vocab = sorted({tok for text in texts for tok in tokenize(text)})
    index = {tok: i for i, tok in enumerate(vocab)}
    matrix = np.zeros((len(texts), len(vocab)))
    for r, text in enumerate(texts):
        for tok in tokenize(text):
            matrix[r, index[tok]] += 1
    return matrix


def nvidia_embed(texts):
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts,
    )
    vectors = [None] * len(texts)
    for item in response.data:
        vectors[item.index] = item.embedding
    return np.array(vectors)


def profile_text(person):
    parts = [
        person.get("major", ""),
        " ".join(person.get("skills", [])),
        " ".join(person.get("experience", [])),
        person.get("location_preference", ""),
    ]
    return clean_text(" ".join(parts))


def job_text(rec):
    parts = [
        rec.get("title", ""),
        rec.get("company", ""),
        rec.get("location", ""),
        " ".join(rec.get("skills") or []),
        rec.get("description", ""),
    ]
    return clean_text(" ".join(str(p) for p in parts))


corpus = [profile_text(student)] + [job_text(rec) for rec in job_records]

if HAS_NVIDIA:
    vectors = nvidia_embed(corpus)
    print("embeddings: NVIDIA")
else:
    vectors = bow_embed(corpus)
    print("embeddings: bag-of-words fallback")

student_vec = vectors[0]
for rec, vec in zip(job_records, vectors[1:]):
    rec["semantic_score"] = cosine(student_vec, vec)

pd.DataFrame(job_records)[["title", "skill_score", "semantic_score"]]


What you should see: a second score column. The Harbor AI role should look better here than it did on keywords if you used NVIDIA embeddings — even the fallback should lift jobs that share Python / ML words.


## 11. Rank Internships

🟦 **CONCEPT** — One number, several signals.

```
Final score
  40% skill match
  30% semantic similarity
  20% experience fit
  10% location
```


🟩 **BUILD** — Score, sort, table.


In [ ]:
def experience_score(person, rec):
    blob = job_text(rec).lower()
    hits = 0
    notes = person.get("experience") or []
    if not notes:
        return 0.0
    for item in notes:
        tokens = tokenize(item)
        if any(tok in blob for tok in tokens if len(tok) > 3):
            hits += 1
    return hits / len(notes)


def location_score(person, rec):
    want = (person.get("location_preference") or "").lower()
    got = (rec.get("location") or "").lower()
    if not want or not got:
        return 0.5
    want_city = want.split(",")[0].strip()
    if want_city and want_city in got:
        return 1.0
    if "remote" in got:
        return 0.7
    return 0.2


for rec in job_records:
    rec["experience_score"] = experience_score(student, rec)
    rec["location_score"] = location_score(student, rec)
    rec["final_score"] = (
        0.40 * rec.get("skill_score", 0)
        + 0.30 * rec.get("semantic_score", 0)
        + 0.20 * rec.get("experience_score", 0)
        + 0.10 * rec.get("location_score", 0)
    )

ranked = sorted(job_records, key=lambda r: r["final_score"], reverse=True)

ranked_df = pd.DataFrame(ranked)[
    ["title", "company", "location", "skill_score", "semantic_score", "final_score"]
]
ranked_df


What you should see: ML roles at the top, SWE / analyst lower. Boston should get a small bump.


## 12. Explain the Match

🟦 **CONCEPT** — A score of 0.87 is not useful by itself.

You want:

```
Why this is a strong match
✓ You have Python
✓ You have PyTorch

Potential gap
⚠ The posting prefers previous research experience
```


🟩 **BUILD** — Ground the explanation in the posting. Fallback is a template from skill overlap.


In [ ]:
def explain_rules(person, rec):
    have = set(s.lower() for s in person.get("skills", []))
    need = [s for s in (rec.get("skills") or [])]
    hits = [s for s in need if s.lower() in have]
    gaps = [s for s in need if s.lower() not in have]
    lines = [f"Match for {rec.get('title')} at {rec.get('company') or 'this company'}."]
    if hits:
        lines.append("You already have: " + ", ".join(hits) + ".")
    if gaps:
        lines.append("Missing or unlisted: " + ", ".join(gaps) + ".")
    else:
        lines.append("No obvious skill gaps from the extracted list.")
    loc = rec.get("location") or "an unlisted location"
    lines.append(f"Location: {loc}.")
    return "\n".join(lines)


def explain_llm(person, rec):
    prompt = f'''You are helping a student read one internship posting.
Only use the posting. If something is not in the posting, say you do not know.

Student:
{json.dumps(person, indent=2)}

Posting:
title: {rec.get("title")}
company: {rec.get("company")}
location: {rec.get("location")}
skills: {rec.get("skills")}
education: {rec.get("education")}
text: {rec.get("description", "")[:2000]}

Write:
Why this is a match (short bullets)
Potential gaps (short bullets)
Do not invent benefits, pay, or deadlines.
'''
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=350,
    )
    return response.choices[0].message.content.strip()


def explain(person, rec):
    if HAS_NVIDIA:
        try:
            return explain_llm(person, rec)
        except Exception as exc:
            return explain_rules(person, rec) + f"\n(LLM failed: {type(exc).__name__})"
    return explain_rules(person, rec)


top = ranked[0]
print("top job:", top["title"], "score", round(top["final_score"], 3))
print()
print(explain(student, top))


## 13. RAG

🟦 **CONCEPT** — The model should not make up facts about an internship.

```
Student question
       ↓
Retriever
       ↓
Relevant job text
       ↓
LLM
       ↓
Answer grounded in the posting
```

Ask things like:

- Why am I a good candidate for this internship?
- What skills am I missing?
- Which of these internships should I prioritize?


🚨 **IMPORTANT** — If the retrieved text does not say it, the answer is "I don't know from the posting."


In [ ]:
def retrieve(question, records, k=3):
    docs = [question] + [job_text(rec) for rec in records]
    if HAS_NVIDIA:
        vecs = nvidia_embed(docs)
    else:
        vecs = bow_embed(docs)
    q = vecs[0]
    scored = []
    for rec, vec in zip(records, vecs[1:]):
        scored.append((cosine(q, vec), rec))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:k]


def answer_from_jobs(question, records, k=3):
    top = retrieve(question, records, k=k)
    context = []
    for score, rec in top:
        context.append(
            f"[{rec.get('title')} | {rec.get('company')} | {rec.get('url')}]\n"
            f"{rec.get('description', '')[:1200]}"
        )
    packed = "\n\n".join(context)

    if not HAS_NVIDIA:
        lines = ["No LLM key, so here are the closest postings:"]
        for score, rec in top:
            lines.append(f"- {rec.get('title')} ({rec.get('company')}) score={score:.3f}")
        return "\n".join(lines), top

    prompt = f'''Answer the student using only the internships below.
If the postings do not say, say you do not know.
Quote company names from the text. Do not invent deadlines or pay.

QUESTION:
{question}

POSTINGS:
{packed}
'''
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=400,
    )
    return response.choices[0].message.content.strip(), top


questions = [
    "Why am I a good candidate for the Machine Learning Engineer Intern role?",
    "What skills am I missing for these internships?",
    "Which of these internships should I prioritize?",
]

rag_answer, rag_hits = answer_from_jobs(questions[0], ranked)
print(rag_answer)
print()
print("retrieved:")
for score, rec in rag_hits:
    print(round(score, 3), rec.get("title"))


🟨 **EXPERIMENT** — Swap in `questions[1]` or `questions[2]` and run the cell again.


## 13b. A Tiny Agent With Web Tools

🟦 **CONCEPT** — An agent is a model that can call functions you wrote.

```
question
  ↓
model picks a tool
  ↓
you run the tool
  ↓
model answers using the tool output
```

The agent does not "have the internet." It has the tools you hand it.


🟩 **BUILD** — Three tools. Same collector and fetch rules as before.


In [ ]:
def search_jobs(query):
    hits = retrieve(query, ranked, k=5)
    return [
        {
            "title": rec.get("title"),
            "company": rec.get("company"),
            "location": rec.get("location"),
            "url": rec.get("url"),
            "score": round(score, 3),
        }
        for score, rec in hits
    ]


def fetch_job_page(url):
    if not str(url).startswith(FAKE_HOST):
        return {"error": "this tool only reads the bundled campus site"}
    page = fetch_page(url)
    return {
        "url": url,
        "status": page["status"],
        "text": page_text(page["html"])[:1500] if page["html"] else "",
    }


TOOLS = {
    "search_jobs": search_jobs,
    "fetch_page": fetch_job_page,
    "answer_from_jobs": lambda q: answer_from_jobs(q, ranked)[0],
}

print("tools:", list(TOOLS))
print(search_jobs("pytorch intern"))


In [ ]:
def run_agent(question):
    if not HAS_NVIDIA:
        hits = search_jobs(question)
        return "No key, so the agent just searched.\n" + json.dumps(hits, indent=2)

    pick_prompt = f'''You can use one tool.
Tools:
- search_jobs(query)
- fetch_page(url)
- answer_from_jobs(question)

Return JSON only:
{{"tool": "name", "arg": "..."}}

Student question: {question}
'''
    choice = chat_json(pick_prompt)
    tool_name = choice.get("tool")
    arg = choice.get("arg", question)
    if tool_name not in TOOLS:
        tool_name = "search_jobs"
        arg = question

    result = TOOLS[tool_name](arg)

    wrap = f'''The student asked: {question}

You called {tool_name} with {arg}
Tool result:
{json.dumps(result, indent=2)[:3000]}

Answer the student. Stay with the tool result. Do not invent jobs.
'''
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": wrap}],
        temperature=0.2,
        max_tokens=350,
    )
    return response.choices[0].message.content.strip()


print(run_agent("Which internship uses PyTorch in Boston?"))


### Login, briefly

Some boards hide listings behind a login. There is a fake portal in `data/mock_portal/`.

Open `login.html`. The form goes to `dashboard.html`. On a real site that jump would only work if the server set a session cookie after checking a password.

🚨 **IMPORTANT** — Do not automate login on LinkedIn, Handshake, or Indeed. That usually breaks their terms, and it is brittle anyway.

Give an agent the web by giving it tools: search, fetch a public page, read a public API. Not by impersonating a logged-in browser.


In [ ]:
portal = Path("data/mock_portal")
print("login page title:")
print(BeautifulSoup((portal / "login.html").read_text(encoding="utf-8"), "html.parser").find("h1").get_text())
print()
print("dashboard, first 300 chars:")
print(page_text((portal / "dashboard.html").read_text(encoding="utf-8"))[:300])


## 14. Evaluation

🟦 **CONCEPT** — An AI system is not finished when it works once. It is finished when you can measure how well it works.

Small labeled set:

| Job | Expected |
|---|---|
| Machine Learning Engineer Intern (Northstar) | strong |
| ML Research Intern (Cedar) | medium |
| Software Engineering Intern (Example Tech) | weak |


🟩 **BUILD** — A few checks. Not a framework. Just questions we can answer.


In [ ]:
labels = {
    "Machine Learning Engineer Intern": "strong",
    "ML Research Intern": "medium",
    "Software Engineering Intern": "weak",
}

eval_rows = [rec for rec in ranked if rec.get("title") in labels]
eval_order = [rec["title"] for rec in eval_rows]
print("ranked order among labeled jobs:")
for title in eval_order:
    print("-", title, labels[title], "score", round(
        next(r["final_score"] for r in eval_rows if r["title"] == title), 3
    ))

expected = [
    "Machine Learning Engineer Intern",
    "ML Research Intern",
    "Software Engineering Intern",
]
order_ok = eval_order[:3] == expected or (
    eval_order
    and eval_order[0] == "Machine Learning Engineer Intern"
    and eval_order[-1] == "Software Engineering Intern"
)
print()
print("ranking looks right:", order_ok)

northstar = next(
    (r for r in ranked if r.get("title") == "Machine Learning Engineer Intern"),
    None,
)
if northstar:
    skills_ok = "Python" in (northstar.get("skills") or [])
    print("extracted Python on Northstar:", skills_ok)

retrieved = retrieve("machine learning intern pytorch", ranked, k=3)
retrieved_titles = [rec.get("title") for _, rec in retrieved]
print("retrieval top 3:", retrieved_titles)
print("retrieval found an ML intern:", any("Machine Learning" in t or "Deep Learning" in t or "ML " in t for t in retrieved_titles))

explanation = explain_rules(student, northstar) if northstar else ""
grounded = "Python" in explanation
print("rule explanation mentions Python:", grounded)
print()
print("hallucination check: explanations must not mention salary unless the posting does.")
print("Northstar posting has salary:", "salary" in job_text(northstar).lower() if northstar else None)


What you should see: Northstar first, SWE last, Python extracted, retrieval returning an ML intern.

If ranking is wrong, look at the weights or the extractor — not the LLM first.


## 15. Final System

Put the pieces you already wrote into one function.


In [ ]:
def find_internships(person, records=None):
    records = records if records is not None else job_records
    scored = []
    person_vec = None

    texts = [profile_text(person)] + [job_text(rec) for rec in records]
    vecs = nvidia_embed(texts) if HAS_NVIDIA else bow_embed(texts)
    person_vec = vecs[0]

    for rec, vec in zip(records, vecs[1:]):
        item = dict(rec)
        item["skill_score"] = skill_match(person.get("skills", []), item.get("skills") or [])
        item["semantic_score"] = cosine(person_vec, vec)
        item["experience_score"] = experience_score(person, item)
        item["location_score"] = location_score(person, item)
        item["final_score"] = (
            0.40 * item["skill_score"]
            + 0.30 * item["semantic_score"]
            + 0.20 * item["experience_score"]
            + 0.10 * item["location_score"]
        )
        scored.append(item)

    scored.sort(key=lambda r: r["final_score"], reverse=True)
    for item in scored:
        item["why"] = explain_rules(person, item)
    return scored


results = find_internships(student)
out = pd.DataFrame(results)[["title", "company", "location", "final_score", "extractor"]]
out


In [ ]:
print("Top match")
print(results[0]["title"], "/", results[0].get("company"))
print()
print(results[0]["why"])


```
                  ┌───────────────┐
                  │ Student       │
                  │ Profile       │
                  └───────┬───────┘
                          │
                          ▼
                  ┌───────────────┐
                  │ Search        │
                  │ internships   │
                  └───────┬───────┘
                          │
                          ▼
                  ┌───────────────┐
                  │ Web Collector │
                  └───────┬───────┘
                          │
                          ▼
                  ┌───────────────┐
                  │ Extract       │
                  │ (rules + LLM) │
                  └───────┬───────┘
                          │
                          ▼
                  ┌───────────────┐
                  │ Job Database  │
                  └───────┬───────┘
                          │
                          ▼
                  ┌───────────────┐
                  │ Embeddings +  │
                  │ Retrieval     │
                  └───────┬───────┘
                          │
                          ▼
                  ┌───────────────┐
                  │ Ranking       │
                  └───────┬───────┘
                          │
                          ▼
                  ┌───────────────┐
                  │ LLM           │
                  │ Explanation   │
                  └───────┬───────┘
                          │
                          ▼
                  Best ML Internships
```

You built the boring system first. Then you added intelligence on top.

### Things to try next

- Set `FETCH_LIVE = True` and add a couple more Greenhouse board tokens
- Tighten `is_job_link` so the blog post never enters the table
- Change the 40 / 30 / 20 / 10 weights and see who rises
- Add 10 labeled jobs and keep score
- Ask the RAG cell a question whose answer is **not** in the postings, and check that it refuses
